
# Methodological effects analysis

This notebook prepares the analytical data for Figure 2.

The scientific objective is not to rank classifiers. The four model families and 100 seeds provide repeated support for estimating how reported AMP benchmark performance changes with:

1. negative-class construction;
2. partition strategy;
3. sequence-similarity/redundancy threshold.

Primary metrics:

- **Average precision (AP)**: threshold-independent ranking performance.
- **MCC**: threshold-dependent binary decision quality.

Primary contrasts:

\[
\Delta_{\mathrm{negative}} = M_{\mathrm{T2}} - M_{\mathrm{T1}}
\]

\[
\Delta_{\mathrm{partition}} = M_{\mathrm{H90}} - M_{\mathrm{RANDOM}}
\]

\[
\Delta_{\mathrm{threshold}} = M_{\mathrm{H30}} - M_{\mathrm{H90}}
\]

Negative values indicate lower performance under the alternative or stricter methodological decision. These are descriptive, seed-index-aligned contrasts, not universal causal effects.


In [ ]:

from __future__ import annotations

from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from itertools import combinations
from pathlib import Path
from typing import Any
import hashlib
import json
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

ANALYSIS_VERSION = "1.0.0"
BOOTSTRAP_RESAMPLES = 10_000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_BASE_SEED = 20260802
BOOTSTRAP_BATCH_SIZE = 1_000

TASK_ORDER = [
    "amp_vs_toxic_without_amp_evidence",
    "amp_vs_without_amp_evidence",
]
TASK_LABELS = {
    TASK_ORDER[0]: "T1",
    TASK_ORDER[1]: "T2",
}

REGIME_ORDER = ["RANDOM", "H90", "H70", "H50", "H30"]
REGIME_LABELS = {
    "RANDOM": "Random",
    "H90": "SIM90",
    "H70": "SIM70",
    "H50": "SIM50",
    "H30": "SIM30",
}

MODEL_ORDER = [
    "knn",
    "logistic_regression",
    "random_forest",
    "linear_svm",
]
MODEL_LABELS = {
    "knn": "KNN",
    "logistic_regression": "LR",
    "random_forest": "RF",
    "linear_svm": "SVM",
}

METRICS = ["average_precision", "mcc"]
METRIC_LABELS = {
    "average_precision": "AP",
    "mcc": "MCC",
}

EXPECTED_SEEDS = 100
EXPECTED_FINAL_RUNS = (
    len(TASK_ORDER)
    * len(REGIME_ORDER)
    * len(MODEL_ORDER)
    * EXPECTED_SEEDS
)


In [ ]:

CURRENT_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name == "notebooks"
    else CURRENT_DIRECTORY
)
if not (REPO_ROOT / "configs").is_dir():
    raise RuntimeError(
        "Run this notebook from the repository root or notebooks directory."
    )
RESULTS_ROOT = REPO_ROOT / "results"
SUMMARY_ROOT = RESULTS_ROOT / "summaries"
ANALYSIS_ROOT = RESULTS_ROOT / "final_analysis"
FIGURE2_ROOT = ANALYSIS_ROOT / "figure2_data"
SUPPLEMENTARY_ROOT = ANALYSIS_ROOT / "supplementary_data"
METADATA_ROOT = REPO_ROOT / "metadata"

PERFORMANCE_PATH = SUMMARY_ROOT / "model_performance.csv"
SELECTED_VARIANTS_PATH = SUMMARY_ROOT / "selected_model_variants.csv"
TRAINING_MANIFEST_PATH = METADATA_ROOT / "model_training_manifest.json"
ANALYSIS_MANIFEST_PATH = (
    METADATA_ROOT / "methodological_effects_analysis_manifest.json"
)

FIGURE2_ROOT.mkdir(parents=True, exist_ok=True)
SUPPLEMENTARY_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT.mkdir(parents=True, exist_ok=True)

if not PERFORMANCE_PATH.is_file():
    raise FileNotFoundError(
        f"Final performance table not found: {PERFORMANCE_PATH}"
    )

print(f"Repository root: {REPO_ROOT}")
print(f"Performance input: {PERFORMANCE_PATH}")
print(f"Figure 2 outputs: {FIGURE2_ROOT}")


## 1. Load and validate the completed experiment

In [ ]:

performance = pd.read_csv(PERFORMANCE_PATH, keep_default_na=True)

required = [
    "task_id",
    "regime_id",
    "seed",
    "model_id",
    "variant_id",
    "partition",
    "average_precision",
    "mcc",
]
missing = [column for column in required if column not in performance.columns]
if missing:
    raise ValueError(
        "model_performance.csv lacks required columns: "
        + ", ".join(missing)
    )

for column in [
    "task_id",
    "regime_id",
    "model_id",
    "variant_id",
    "partition",
]:
    performance[column] = performance[column].astype(str).str.strip()

performance["seed"] = pd.to_numeric(
    performance["seed"],
    errors="raise",
).astype(int)

for metric in METRICS:
    performance[metric] = pd.to_numeric(
        performance[metric],
        errors="raise",
    )
    if not np.isfinite(performance[metric].to_numpy(dtype=float)).all():
        raise ValueError(f"Non-finite values found in {metric}.")

if set(performance["task_id"].unique()) != set(TASK_ORDER):
    raise ValueError("Unexpected tasks in model_performance.csv.")

if set(performance["regime_id"].unique()) != set(REGIME_ORDER):
    raise ValueError("Unexpected regimes in model_performance.csv.")

if set(performance["model_id"].unique()) != set(MODEL_ORDER):
    raise ValueError("Unexpected model families in model_performance.csv.")

if set(performance["partition"].str.lower().unique()) != {
    "validation",
    "test",
}:
    raise ValueError("Expected exactly validation and test rows.")

run_keys = [
    "task_id",
    "regime_id",
    "seed",
    "model_id",
    "variant_id",
]

if performance.duplicated(run_keys + ["partition"]).any():
    raise ValueError("Duplicated validation/test rows detected.")

partition_counts = performance.groupby(
    run_keys,
    observed=True,
)["partition"].nunique()

if not (partition_counts == 2).all():
    raise ValueError(
        "Every final run must contain one validation and one test row."
    )

test_results = performance[
    performance["partition"].str.lower().eq("test")
].copy()

validation_results = performance[
    performance["partition"].str.lower().eq("validation")
].copy()

if len(test_results) != EXPECTED_FINAL_RUNS:
    raise ValueError(
        f"Expected {EXPECTED_FINAL_RUNS:,} test runs; "
        f"found {len(test_results):,}."
    )

if len(validation_results) != EXPECTED_FINAL_RUNS:
    raise ValueError(
        f"Expected {EXPECTED_FINAL_RUNS:,} validation runs; "
        f"found {len(validation_results):,}."
    )

seed_counts = test_results.groupby(
    ["task_id", "regime_id", "model_id"],
    observed=True,
)["seed"].nunique()

if not (seed_counts == EXPECTED_SEEDS).all():
    display(seed_counts[seed_counts != EXPECTED_SEEDS])
    raise ValueError("Incomplete seed matrix detected.")

variant_check = test_results.groupby(
    ["task_id", "model_id"],
    observed=True,
).agg(
    n_variants=("variant_id", "nunique"),
    variant_id=(
        "variant_id",
        lambda values: " | ".join(sorted(set(values))),
    ),
    n_regimes=("regime_id", "nunique"),
    n_seeds=("seed", "nunique"),
).reset_index()

if not (
    (variant_check["n_variants"] == 1)
    & (variant_check["n_regimes"] == len(REGIME_ORDER))
    & (variant_check["n_seeds"] == EXPECTED_SEEDS)
).all():
    display(variant_check)
    raise ValueError("Model variants are not frozen across regimes.")

print(f"Validated test runs: {len(test_results):,}")
print(f"Validated validation runs: {len(validation_results):,}")
display(variant_check)


## 2. Long-form test data and statistical utilities

In [ ]:

test_long = test_results.melt(
    id_vars=[
        "task_id",
        "regime_id",
        "seed",
        "model_id",
        "variant_id",
    ],
    value_vars=METRICS,
    var_name="metric",
    value_name="value",
)

test_long["task_label"] = test_long["task_id"].map(TASK_LABELS)
test_long["regime_label"] = test_long["regime_id"].map(REGIME_LABELS)
test_long["model_label"] = test_long["model_id"].map(MODEL_LABELS)
test_long["metric_label"] = test_long["metric"].map(METRIC_LABELS)


def stable_seed(base_seed: int, *parts: Any) -> int:
    material = "|".join(
        [str(base_seed), *map(str, parts)]
    ).encode("utf-8")
    digest = hashlib.sha256(material).digest()
    return int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False,
    ) % (2**32)


def bootstrap_median_ci(
    values: np.ndarray,
    *,
    seed: int,
) -> tuple[float, float]:
    values = np.asarray(values, dtype=np.float64)

    if values.ndim != 1 or values.size == 0:
        raise ValueError("Bootstrap input must be one-dimensional.")

    rng = np.random.default_rng(seed)
    medians = np.empty(BOOTSTRAP_RESAMPLES, dtype=np.float64)
    offset = 0

    while offset < BOOTSTRAP_RESAMPLES:
        current = min(
            BOOTSTRAP_BATCH_SIZE,
            BOOTSTRAP_RESAMPLES - offset,
        )
        indices = rng.integers(
            0,
            values.size,
            size=(current, values.size),
        )
        medians[offset:offset + current] = np.median(
            values[indices],
            axis=1,
        )
        offset += current

    alpha = 1.0 - BOOTSTRAP_CONFIDENCE_LEVEL

    return (
        float(np.quantile(medians, alpha / 2.0)),
        float(np.quantile(medians, 1.0 - alpha / 2.0)),
    )


def summarize_values(
    values: np.ndarray,
    *,
    seed: int,
) -> dict[str, Any]:
    values = np.asarray(values, dtype=np.float64)

    q1 = float(np.quantile(values, 0.25))
    median = float(np.quantile(values, 0.50))
    q3 = float(np.quantile(values, 0.75))
    ci_low, ci_high = bootstrap_median_ci(values, seed=seed)

    return {
        "n": int(values.size),
        "mean": float(np.mean(values)),
        "standard_deviation": float(np.std(values, ddof=1)),
        "minimum": float(np.min(values)),
        "q1": q1,
        "median": median,
        "q3": q3,
        "iqr": q3 - q1,
        "maximum": float(np.max(values)),
        "bootstrap_median_ci_low": ci_low,
        "bootstrap_median_ci_high": ci_high,
    }


def grouped_summary(
    dataframe: pd.DataFrame,
    *,
    groups: list[str],
    namespace: str,
) -> pd.DataFrame:
    rows = []

    for key, group in dataframe.groupby(
        groups,
        observed=True,
        sort=True,
    ):
        if not isinstance(key, tuple):
            key = (key,)

        record = dict(zip(groups, key))
        seed = stable_seed(
            BOOTSTRAP_BASE_SEED,
            namespace,
            *key,
        )

        rows.append(
            {
                **record,
                **summarize_values(
                    group["value"].to_numpy(dtype=float),
                    seed=seed,
                ),
            }
        )

    return pd.DataFrame(rows)


## 3. Panel B — generalisation trajectories

In [ ]:

trajectory_seed_level = test_long.groupby(
    [
        "task_id",
        "task_label",
        "regime_id",
        "regime_label",
        "seed",
        "metric",
        "metric_label",
    ],
    observed=True,
    as_index=False,
).agg(
    value=("value", "median"),
    n_models=("model_id", "nunique"),
)

if not (trajectory_seed_level["n_models"] == len(MODEL_ORDER)).all():
    raise ValueError("Trajectory values do not include all models.")

trajectory_summary = grouped_summary(
    trajectory_seed_level,
    groups=[
        "task_id",
        "task_label",
        "regime_id",
        "regime_label",
        "metric",
        "metric_label",
    ],
    namespace="trajectory",
)

display(trajectory_summary)


## 4. Panel A — effect of negative-class construction

In [ ]:

task_wide = test_long.pivot(
    index=[
        "regime_id",
        "regime_label",
        "model_id",
        "model_label",
        "seed",
        "metric",
        "metric_label",
    ],
    columns="task_id",
    values="value",
).reset_index()

task_wide.columns.name = None

negative_model_level = task_wide[
    [
        "regime_id",
        "regime_label",
        "model_id",
        "model_label",
        "seed",
        "metric",
        "metric_label",
    ]
].copy()

negative_model_level["value"] = (
    task_wide[TASK_ORDER[1]]
    - task_wide[TASK_ORDER[0]]
)
negative_model_level["effect_id"] = "negative_class_construction"
negative_model_level["effect_label"] = "Negative-class construction"
negative_model_level["contrast"] = "T2 - T1"
negative_model_level["reference"] = "T1"
negative_model_level["comparison"] = "T2"

negative_seed_level = negative_model_level.groupby(
    [
        "regime_id",
        "regime_label",
        "seed",
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
    ],
    observed=True,
    as_index=False,
).agg(
    value=("value", "median"),
    n_models=("model_id", "nunique"),
)

if not (negative_seed_level["n_models"] == len(MODEL_ORDER)).all():
    raise ValueError("Negative-class contrasts lack model coverage.")

negative_summary = grouped_summary(
    negative_seed_level,
    groups=[
        "regime_id",
        "regime_label",
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
    ],
    namespace="negative_class",
)

display(negative_summary)


## 5. Full partition and redundancy-threshold contrasts

In [ ]:

regime_wide = test_long.pivot(
    index=[
        "task_id",
        "task_label",
        "model_id",
        "model_label",
        "seed",
        "metric",
        "metric_label",
    ],
    columns="regime_id",
    values="value",
).reset_index()

regime_wide.columns.name = None


def build_regime_contrasts(
    comparison_regimes: list[str],
    *,
    reference_regime: str,
    effect_id: str,
    effect_label: str,
) -> pd.DataFrame:
    frames = []

    for comparison_regime in comparison_regimes:
        current = regime_wide[
            [
                "task_id",
                "task_label",
                "model_id",
                "model_label",
                "seed",
                "metric",
                "metric_label",
            ]
        ].copy()

        current["regime_id"] = comparison_regime
        current["regime_label"] = REGIME_LABELS[comparison_regime]
        current["value"] = (
            regime_wide[comparison_regime]
            - regime_wide[reference_regime]
        )
        current["effect_id"] = effect_id
        current["effect_label"] = effect_label
        current["contrast"] = (
            f"{comparison_regime} - {reference_regime}"
        )
        current["reference"] = reference_regime
        current["comparison"] = comparison_regime
        frames.append(current)

    return pd.concat(frames, ignore_index=True)


partition_model_level = build_regime_contrasts(
    ["H90", "H70", "H50", "H30"],
    reference_regime="RANDOM",
    effect_id="partition_strategy",
    effect_label="Partition strategy",
)

threshold_model_level = build_regime_contrasts(
    ["H70", "H50", "H30"],
    reference_regime="H90",
    effect_id="redundancy_threshold",
    effect_label="Redundancy threshold",
)


def aggregate_models(dataframe: pd.DataFrame) -> pd.DataFrame:
    return dataframe.groupby(
        [
            "task_id",
            "task_label",
            "regime_id",
            "regime_label",
            "seed",
            "metric",
            "metric_label",
            "effect_id",
            "effect_label",
            "contrast",
            "reference",
            "comparison",
        ],
        observed=True,
        as_index=False,
    ).agg(
        value=("value", "median"),
        n_models=("model_id", "nunique"),
    )


partition_seed_level = aggregate_models(partition_model_level)
threshold_seed_level = aggregate_models(threshold_model_level)

partition_summary = grouped_summary(
    partition_seed_level,
    groups=[
        "task_id",
        "task_label",
        "regime_id",
        "regime_label",
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
    ],
    namespace="partition",
)

threshold_summary = grouped_summary(
    threshold_seed_level,
    groups=[
        "task_id",
        "task_label",
        "regime_id",
        "regime_label",
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
    ],
    namespace="threshold",
)

display(partition_summary)
display(threshold_summary)


## 6. Panel C — pre-specified methodological effects

In [ ]:

principal_negative = negative_seed_level[
    negative_seed_level["regime_id"].eq("RANDOM")
][
    [
        "seed",
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
        "value",
    ]
].copy()

principal_partition = partition_seed_level[
    partition_seed_level["regime_id"].eq("H90")
].groupby(
    [
        "seed",
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
    ],
    observed=True,
    as_index=False,
).agg(
    value=("value", "median"),
    n_tasks=("task_id", "nunique"),
)

principal_threshold = threshold_seed_level[
    threshold_seed_level["regime_id"].eq("H30")
].groupby(
    [
        "seed",
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
    ],
    observed=True,
    as_index=False,
).agg(
    value=("value", "median"),
    n_tasks=("task_id", "nunique"),
)

if not (principal_partition["n_tasks"] == len(TASK_ORDER)).all():
    raise ValueError("Partition principal effect lacks both tasks.")

if not (principal_threshold["n_tasks"] == len(TASK_ORDER)).all():
    raise ValueError("Threshold principal effect lacks both tasks.")

principal_columns = principal_negative.columns

principal_effects = pd.concat(
    [
        principal_negative,
        principal_partition[principal_columns],
        principal_threshold[principal_columns],
    ],
    ignore_index=True,
)

principal_effect_summary = grouped_summary(
    principal_effects,
    groups=[
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
    ],
    namespace="principal_effect",
)

display(principal_effect_summary)


## 7. Panel D — effect magnitude versus model-family variation

In [ ]:

model_variation_rows = []

for key, group in test_long.groupby(
    [
        "task_id",
        "task_label",
        "regime_id",
        "regime_label",
        "seed",
        "metric",
        "metric_label",
    ],
    observed=True,
    sort=True,
):
    (
        task_id,
        task_label,
        regime_id,
        regime_label,
        seed,
        metric,
        metric_label,
    ) = key

    model_values = dict(
        zip(group["model_id"], group["value"])
    )

    if set(model_values) != set(MODEL_ORDER):
        raise ValueError("Model-family variation lacks a model.")

    pairwise = [
        abs(
            float(model_values[first])
            - float(model_values[second])
        )
        for first, second in combinations(MODEL_ORDER, 2)
    ]

    model_variation_rows.append(
        {
            "task_id": task_id,
            "task_label": task_label,
            "regime_id": regime_id,
            "regime_label": regime_label,
            "seed": int(seed),
            "metric": metric,
            "metric_label": metric_label,
            "value": float(np.median(pairwise)),
            "n_model_pairs": len(pairwise),
        }
    )

model_variation_condition = pd.DataFrame(model_variation_rows)

model_variation_seed = model_variation_condition.groupby(
    ["seed", "metric", "metric_label"],
    observed=True,
    as_index=False,
).agg(
    value=("value", "median"),
    n_conditions=("task_id", "size"),
)

expected_conditions = len(TASK_ORDER) * len(REGIME_ORDER)

if not (model_variation_seed["n_conditions"] == expected_conditions).all():
    raise ValueError("Model-family variation lacks task-regime conditions.")

model_variation_seed["effect_id"] = "model_family"
model_variation_seed["effect_label"] = "Model family"
model_variation_seed["contrast"] = (
    "Median absolute pairwise model difference"
)
model_variation_seed["reference"] = "Fixed benchmark and split"
model_variation_seed["comparison"] = "Alternative model family"

methodological_magnitude = principal_effects.copy()
methodological_magnitude["signed_value"] = methodological_magnitude["value"]
methodological_magnitude["value"] = methodological_magnitude["value"].abs()

model_magnitude = model_variation_seed[
    [
        "seed",
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
        "value",
    ]
].copy()
model_magnitude["signed_value"] = np.nan

magnitude_columns = model_magnitude.columns

effect_magnitude = pd.concat(
    [
        methodological_magnitude[magnitude_columns],
        model_magnitude,
    ],
    ignore_index=True,
)

effect_magnitude_summary = grouped_summary(
    effect_magnitude,
    groups=[
        "metric",
        "metric_label",
        "effect_id",
        "effect_label",
        "contrast",
        "reference",
        "comparison",
    ],
    namespace="effect_magnitude",
)

display(effect_magnitude_summary)


## 8. Supplementary diagnostics

In [ ]:

pair_keys = [
    "task_id",
    "regime_id",
    "seed",
    "model_id",
    "variant_id",
]

validation_pair = validation_results[
    pair_keys + METRICS
].rename(
    columns={
        metric: f"{metric}_validation"
        for metric in METRICS
    }
)

test_pair = test_results[
    pair_keys + METRICS
].rename(
    columns={
        metric: f"{metric}_test"
        for metric in METRICS
    }
)

paired = validation_pair.merge(
    test_pair,
    on=pair_keys,
    validate="one_to_one",
)

optimism_frames = []

for metric in METRICS:
    current = paired[pair_keys].copy()
    current["metric"] = metric
    current["metric_label"] = METRIC_LABELS[metric]
    current["value"] = (
        paired[f"{metric}_validation"]
        - paired[f"{metric}_test"]
    )
    optimism_frames.append(current)

validation_optimism_model = pd.concat(
    optimism_frames,
    ignore_index=True,
)

validation_optimism_seed = validation_optimism_model.groupby(
    [
        "task_id",
        "regime_id",
        "seed",
        "metric",
        "metric_label",
    ],
    observed=True,
    as_index=False,
).agg(
    value=("value", "median"),
    n_models=("model_id", "nunique"),
)

validation_optimism_seed["task_label"] = (
    validation_optimism_seed["task_id"].map(TASK_LABELS)
)
validation_optimism_seed["regime_label"] = (
    validation_optimism_seed["regime_id"].map(REGIME_LABELS)
)

validation_optimism_summary = grouped_summary(
    validation_optimism_seed,
    groups=[
        "task_id",
        "task_label",
        "regime_id",
        "regime_label",
        "metric",
        "metric_label",
    ],
    namespace="validation_optimism",
)

composition_columns = [
    column
    for column in [
        "n",
        "positive_count",
        "negative_count",
        "prevalence",
    ]
    if column in test_results.columns
]

if composition_columns:
    composition_seed = test_results.sort_values(
        ["task_id", "regime_id", "seed", "model_id"]
    ).drop_duplicates(
        ["task_id", "regime_id", "seed"]
    )[
        [
            "task_id",
            "regime_id",
            "seed",
            *composition_columns,
        ]
    ].copy()

    composition_seed["task_label"] = (
        composition_seed["task_id"].map(TASK_LABELS)
    )
    composition_seed["regime_label"] = (
        composition_seed["regime_id"].map(REGIME_LABELS)
    )
else:
    composition_seed = pd.DataFrame()

display(validation_optimism_summary)


## 9. Export analytical datasets

In [ ]:

outputs = {
    "panel_a_seed": FIGURE2_ROOT
    / "panel_a_negative_class_effect_seed_level.csv",
    "panel_a_summary": FIGURE2_ROOT
    / "panel_a_negative_class_effect_summary.csv",
    "panel_b_seed": FIGURE2_ROOT
    / "panel_b_generalisation_trajectories_seed_level.csv",
    "panel_b_summary": FIGURE2_ROOT
    / "panel_b_generalisation_trajectories_summary.csv",
    "panel_c_seed": FIGURE2_ROOT
    / "panel_c_methodological_effects_seed_level.csv",
    "panel_c_summary": FIGURE2_ROOT
    / "panel_c_methodological_effects_summary.csv",
    "panel_d_seed": FIGURE2_ROOT
    / "panel_d_effect_magnitudes_vs_model_family_seed_level.csv",
    "panel_d_summary": FIGURE2_ROOT
    / "panel_d_effect_magnitudes_vs_model_family_summary.csv",
    "partition_seed": SUPPLEMENTARY_ROOT
    / "partition_effects_seed_level.csv",
    "partition_summary": SUPPLEMENTARY_ROOT
    / "partition_effects_summary.csv",
    "threshold_seed": SUPPLEMENTARY_ROOT
    / "threshold_effects_seed_level.csv",
    "threshold_summary": SUPPLEMENTARY_ROOT
    / "threshold_effects_summary.csv",
    "model_variation_condition": SUPPLEMENTARY_ROOT
    / "model_family_variation_condition_level.csv",
    "model_variation_seed": SUPPLEMENTARY_ROOT
    / "model_family_variation_seed_level.csv",
    "validation_optimism_seed": SUPPLEMENTARY_ROOT
    / "validation_optimism_seed_level.csv",
    "validation_optimism_summary": SUPPLEMENTARY_ROOT
    / "validation_optimism_summary.csv",
    "variant_check": SUPPLEMENTARY_ROOT
    / "model_variant_consistency.csv",
}

negative_seed_level.to_csv(outputs["panel_a_seed"], index=False)
negative_summary.to_csv(outputs["panel_a_summary"], index=False)
trajectory_seed_level.to_csv(outputs["panel_b_seed"], index=False)
trajectory_summary.to_csv(outputs["panel_b_summary"], index=False)
principal_effects.to_csv(outputs["panel_c_seed"], index=False)
principal_effect_summary.to_csv(outputs["panel_c_summary"], index=False)
effect_magnitude.to_csv(outputs["panel_d_seed"], index=False)
effect_magnitude_summary.to_csv(outputs["panel_d_summary"], index=False)
partition_seed_level.to_csv(outputs["partition_seed"], index=False)
partition_summary.to_csv(outputs["partition_summary"], index=False)
threshold_seed_level.to_csv(outputs["threshold_seed"], index=False)
threshold_summary.to_csv(outputs["threshold_summary"], index=False)
model_variation_condition.to_csv(
    outputs["model_variation_condition"],
    index=False,
)
model_variation_seed.to_csv(
    outputs["model_variation_seed"],
    index=False,
)
validation_optimism_seed.to_csv(
    outputs["validation_optimism_seed"],
    index=False,
)
validation_optimism_summary.to_csv(
    outputs["validation_optimism_summary"],
    index=False,
)
variant_check.to_csv(outputs["variant_check"], index=False)

if not composition_seed.empty:
    composition_path = (
        SUPPLEMENTARY_ROOT / "test_set_composition_seed_level.csv"
    )
    composition_seed.to_csv(composition_path, index=False)
    outputs["composition_seed"] = composition_path

for name, path in outputs.items():
    if not path.is_file():
        raise RuntimeError(f"Output not created: {name} -> {path}")

print("Exported analytical datasets:")
for name, path in outputs.items():
    print(f"  {name}: {path.relative_to(REPO_ROOT)}")


## 10. Immediate scientific overview

In [ ]:

print("Panel A — benchmark-definition effect")
display(
    negative_summary[
        [
            "regime_label",
            "metric_label",
            "n",
            "median",
            "q1",
            "q3",
            "iqr",
            "bootstrap_median_ci_low",
            "bootstrap_median_ci_high",
        ]
    ]
)

print("Panel B — trajectories")
display(
    trajectory_summary[
        [
            "task_label",
            "regime_label",
            "metric_label",
            "n",
            "median",
            "q1",
            "q3",
            "iqr",
            "bootstrap_median_ci_low",
            "bootstrap_median_ci_high",
        ]
    ]
)

print("Panel C — principal methodological effects")
display(
    principal_effect_summary[
        [
            "effect_label",
            "contrast",
            "metric_label",
            "n",
            "median",
            "q1",
            "q3",
            "iqr",
            "bootstrap_median_ci_low",
            "bootstrap_median_ci_high",
        ]
    ]
)

print("Panel D — absolute effect magnitude")
display(
    effect_magnitude_summary[
        [
            "effect_label",
            "metric_label",
            "n",
            "median",
            "q1",
            "q3",
            "iqr",
            "bootstrap_median_ci_low",
            "bootstrap_median_ci_high",
        ]
    ]
)


## 11. Analysis manifest

In [ ]:

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def package_version(package_name: str) -> str:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not-installed"


def git_value(arguments: list[str]) -> str | None:
    try:
        completed = subprocess.run(
            ["git", *arguments],
            cwd=REPO_ROOT,
            check=True,
            capture_output=True,
            text=True,
        )
        return completed.stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


manifest = {
    "schema_version": "1.0",
    "analysis_version": ANALYSIS_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": (
        "Quantify sensitivity of AMP benchmark performance "
        "to methodological decisions."
    ),
    "scope": {
        "analysis_type": "descriptive demonstrative analysis",
        "models_retrained": False,
        "hypothesis_tests_performed": False,
        "tasks": TASK_ORDER,
        "regimes": REGIME_ORDER,
        "models": MODEL_ORDER,
        "seeds": EXPECTED_SEEDS,
        "metrics": METRICS,
    },
    "effect_definitions": {
        "negative_class_construction": {
            "formula": "T2 - T1",
            "primary_regime": "RANDOM",
        },
        "partition_strategy": {
            "formula": "H90 - RANDOM",
        },
        "redundancy_threshold": {
            "formula": "H30 - H90",
        },
        "model_family": {
            "formula": (
                "median absolute pairwise model difference "
                "under fixed task, regime, and seed"
            ),
        },
    },
    "bootstrap": {
        "estimand": "median",
        "method": "percentile bootstrap",
        "resamples": BOOTSTRAP_RESAMPLES,
        "confidence_level": BOOTSTRAP_CONFIDENCE_LEVEL,
        "base_seed": BOOTSTRAP_BASE_SEED,
    },
    "inputs": {
        "model_performance": {
            "path": str(PERFORMANCE_PATH.relative_to(REPO_ROOT)),
            "sha256": sha256_file(PERFORMANCE_PATH),
        },
    },
    "outputs": {
        name: {
            "path": str(path.relative_to(REPO_ROOT)),
            "sha256": sha256_file(path),
        }
        for name, path in outputs.items()
    },
    "git": {
        "commit": git_value(["rev-parse", "HEAD"]),
        "status_porcelain": git_value(["status", "--porcelain"]),
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "jupyter": package_version("jupyter"),
        "ipython": package_version("ipython"),
    },
}

with ANALYSIS_MANIFEST_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    handle.write("\n")

print(
    "Analysis manifest:",
    ANALYSIS_MANIFEST_PATH.relative_to(REPO_ROOT),
)



## Handoff to Notebook 2

Notebook 2 should consume only:

```text
results/final_analysis/figure2_data/
```

Planned panels:

- **A:** distribution of the negative-class construction effect across regimes;
- **B:** model-aggregated AP and MCC trajectories for T1 and T2;
- **C:** signed effects of the three pre-specified methodological decisions;
- **D:** absolute methodological effect magnitudes compared with typical model-family variation.

The plotting notebook should not recompute contrasts or statistical summaries.
